# Phase 6 setup — CSE-CIC-IDS2018 (improved): CSV → Parquet — **v2 (A1.0 fix)**

**Notebook:** `notebooks/00_setup/05_convert_cse2018.ipynb`
**Raw data:** `data/raw/cicids2017_improved/CSECICIDS2018_improved/` (10 day CSVs)
**Pre-registration:** `reports/advisor_notes/PREREG_cse2018.md` + **Addendum A1**.

**What changed in v2 (deviation A1.0):** the improved-release column
`attempted_category` is label metadata, not a flow feature; in v1 it slipped
into the feature matrix. It is now in the leakage drop list. Section 5a
deletes the stale v1 processed parquets so Stage 2 regenerates them
(Stage 2 skips existing files, so deletion is mandatory for the fix to land).
Interim parquets are untouched — the re-run is fast (no CSV parsing).

Conventions (identical to `03_csv_to_parquet.ipynb`):
**Stage 1 (interim):** lossless — normalize column names, inf→NaN, Parquet per day.
**Stage 2 (processed):** schema contract — binary `label` (BENIGN→0, all attacks incl. `*- Attempted`→1), `attack_category`, `attempted` bool, `source_file`; drop leakage columns; drop NaN rows; float32; **pre-registered per-day cap 1.5M rows (stratified, seed 42)**.

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/MyDrive/phd_thesis')

%pip install -q pyarrow tqdm

Mounted at /content/drive


In [2]:
import re, time, json, datetime
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

ROOT = Path('/content/drive/MyDrive/phd_thesis')
RAW_2018 = ROOT / 'data' / 'raw' / 'cicids2017_improved' / 'CSECICIDS2018_improved'
INTERIM = ROOT / 'data' / 'interim' / 'cse_cic_ids2018'
PROCESSED = ROOT / 'data' / 'processed' / 'cse_cic_ids2018'
REPORT_PATH = ROOT / 'reports' / 'advisor_notes' / 'conversion_report_cse2018.md'
DATA_CARD_PATH = ROOT / 'reports' / 'advisor_notes' / 'data_card_cse2018.md'

for p in [INTERIM, PROCESSED]:
    p.mkdir(parents=True, exist_ok=True)

SEED = 42                       # pre-registered (PREREG_cse2018.md §2)
MAX_ROWS_PER_DAY = 1_500_000    # pre-registered per-day stratified cap
CHUNK = 1_000_000               # CSV chunk size (memory safety on Colab)

REPORT = {'generated': datetime.datetime.now().isoformat(timespec='seconds'),
          'version': 'v2_A1.0_attempted_category_dropped',
          'seed': SEED, 'max_rows_per_day': MAX_ROWS_PER_DAY, 'days': {}}

print('RAW_2018 =', RAW_2018)
print('files:', sorted(f.name for f in RAW_2018.glob('*.csv')))

RAW_2018 = /content/drive/MyDrive/phd_thesis/data/raw/cicids2017_improved/CSECICIDS2018_improved
files: ['Friday-02-03-2018.csv', 'Friday-16-02-2018.csv', 'Friday-23-02-2018.csv', 'Thursday-01-03-2018.csv', 'Thursday-15-02-2018.csv', 'Thursday-22-02-2018.csv', 'Tuesday-20-02-2018.csv', 'Wednesday-14-02-2018.csv', 'Wednesday-21-02-2018.csv', 'Wednesday-28-02-2018.csv']


## 2. Day mapping and helpers

`normalize_column` and the leakage list are copied from notebook 03 so both
datasets share one convention. **v2 change: `attempted_category` added to the
leakage list** — it is improved-release label metadata (which flows were
attempted-but-unsuccessful attacks), i.e. derived from labels, and must never
reach a detector or the geometry.

In [3]:
# Raw file -> canonical day name (chronological). Dominant attacks noted
# for the drift narrative (Addendum A1.2).
FILE_TO_DAY = {
    'Wednesday-14-02-2018.csv': 'wed_14_02',  # FTP-BruteForce, SSH-Bruteforce
    'Thursday-15-02-2018.csv':  'thu_15_02',  # DoS GoldenEye, Slowloris
    'Friday-16-02-2018.csv':    'fri_16_02',  # DoS SlowHTTPTest, Hulk
    'Tuesday-20-02-2018.csv':   'tue_20_02',  # DDoS LOIC-HTTP, LOIC-UDP
    'Wednesday-21-02-2018.csv': 'wed_21_02',  # DDOS HOIC, LOIC-UDP
    'Thursday-22-02-2018.csv':  'thu_22_02',  # Brute Force -Web/-XSS, SQLi
    'Friday-23-02-2018.csv':    'fri_23_02',  # Brute Force -Web/-XSS, SQLi
    'Wednesday-28-02-2018.csv': 'wed_28_02',  # Infiltration
    'Thursday-01-03-2018.csv':  'thu_01_03',  # Infiltration
    'Friday-02-03-2018.csv':    'fri_02_03',  # Bot
}
DAY_ORDER = list(FILE_TO_DAY.values())

# --- copied from 03_csv_to_parquet.ipynb; v2 adds attempted_category --------
def normalize_column(name: str) -> str:
    name = name.strip()
    name = re.sub(r'[^A-Za-z0-9]+', '_', name)
    name = re.sub(r'_+', '_', name).strip('_')
    return name.lower()

LEAKAGE_COLUMN_PATTERNS = [
    'flow_id', 'src_ip', 'dst_ip', 'source_ip', 'destination_ip',
    'src_port', 'dst_port', 'source_port', 'destination_port',
    'timestamp', 'id',
    'srcip', 'dstip', 'sport', 'dsport', 'stime', 'ltime',
    'attempted_category',   # v2 / A1.0: improved-2018 label metadata
]
# ---------------------------------------------------------------------------

ATTEMPTED_RE = re.compile(r'\s*-\s*attempted\s*$', re.IGNORECASE)

def parse_label(raw: pd.Series):
    """'DoS Hulk - Attempted' -> (category 'DoS Hulk', attempted True).
    Binary rule identical to notebook 03: BENIGN -> 0, everything else
    (including Attempted) -> 1."""
    s = raw.astype(str).str.strip()
    attempted = s.str.contains(ATTEMPTED_RE)
    category = s.str.replace(ATTEMPTED_RE, '', regex=True).str.strip()
    label = (category.str.upper() != 'BENIGN').astype('int8')
    return category, attempted, label

## 3. Stage 1 — CSV → interim Parquet (lossless, chunked)

Skip-if-exists: on the v2 re-run every day prints `[skip]` — that is correct,
interim is unchanged by A1.0. Only delete interim files if the raw CSVs
themselves changed.

In [4]:
interim_stats = []
for fname, day in FILE_TO_DAY.items():
    csv = RAW_2018 / fname
    out = INTERIM / f'{day}.parquet'
    if out.exists():
        print(f'[skip] {out.name} exists'); continue
    if not csv.exists():
        print(f'!! MISSING {fname}'); continue

    t0 = time.time(); chunks = []
    for chunk in pd.read_csv(csv, low_memory=False, chunksize=CHUNK):
        chunk.columns = [normalize_column(cn) for cn in chunk.columns]
        num_cols = chunk.select_dtypes(include=['float64']).columns
        chunk[num_cols] = chunk[num_cols].astype('float32')
        int_cols = chunk.select_dtypes(include=['int64']).columns
        chunk[int_cols] = chunk[int_cols].astype('int32')
        chunks.append(chunk)
    df = pd.concat(chunks, ignore_index=True); del chunks
    df = df.replace([np.inf, -np.inf], np.nan)
    df['source_file'] = fname
    df.to_parquet(out, compression='snappy', index=False)
    print(f'{day}: {df.shape} in {time.time()-t0:.0f}s -> {out.name}')
    interim_stats.append({'day': day, 'rows': len(df), 'cols': df.shape[1]})
    del df

[skip] wed_14_02.parquet exists
[skip] thu_15_02.parquet exists
[skip] fri_16_02.parquet exists
[skip] tue_20_02.parquet exists
[skip] wed_21_02.parquet exists
[skip] thu_22_02.parquet exists
[skip] fri_23_02.parquet exists
[skip] wed_28_02.parquet exists
[skip] thu_01_03.parquet exists
[skip] fri_02_03.parquet exists


## 4. Schema verification against CICIDS2017 — check before proceeding

Diffs the harmonized 2018 features (post leakage-drop, so `attempted_category`
is now excluded here too) against the real processed 2017 schema. The assert
stops the notebook if any 2017 feature is missing from 2018.

In [5]:
cic17_cols = set(pd.read_parquet(
    ROOT / 'data' / 'processed' / 'cicids2017' / 'monday.parquet').columns)
cic17_features = cic17_cols - {'label', 'attack_category', 'attempted', 'source_file'}

probe = pd.read_parquet(INTERIM / 'wed_14_02.parquet')
label_col = [cn for cn in probe.columns if cn == 'label'] or \
            [cn for cn in probe.columns if 'label' in cn]
probe_features = set(probe.columns) - set(label_col) - {'source_file'} \
                 - set(LEAKAGE_COLUMN_PATTERNS)

only_2018 = sorted(probe_features - cic17_features)
only_2017 = sorted(cic17_features - probe_features)
print(f'2017 features: {len(cic17_features)} | 2018 features (post leakage-drop): {len(probe_features)}')
print(f'shared: {len(probe_features & cic17_features)}')
print(f'2018-only: {only_2018}')
print(f'2017-only (MUST be empty or explained): {only_2017}')
REPORT['schema_check'] = {'shared': len(probe_features & cic17_features),
                          'only_2018': only_2018, 'only_2017': only_2017}
assert 'attempted_category' not in probe_features, 'A1.0 fix not active in this kernel — re-run Section 2.'
assert not only_2017, 'Schema mismatch: investigate before Stage 2 (see markdown above).'

2017 features: 82 | 2018 features (post leakage-drop): 82
shared: 82
2018-only: []
2017-only (MUST be empty or explained): []


## 5a. **v2 cleanup — delete stale v1 processed parquets**

Stage 2 skips existing files, so the A1.0 fix only lands if the v1 outputs
are removed first. This deletes ONLY `data/processed/cse_cic_ids2018/*.parquet`
(never interim, never other datasets). Idempotent: on later re-runs it finds
nothing to delete.

In [6]:
stale = sorted(PROCESSED.glob('*.parquet'))
if not stale:
    print('no processed parquets to delete — clean state')
for p in stale:
    print('deleting', p.name)
    p.unlink()
# also catch Drive duplicate copies like 'wed_14_02 (1).parquet'
for p in sorted(PROCESSED.glob('*')):
    print('WARNING — unexpected file still present:', p.name)

no processed parquets to delete — clean state


## 5. Stage 2 — interim → processed (schema contract + pre-registered cap)

Per-day cap: 1.5M rows, stratified by attack_category, seed 42 (pre-registered;
do not change). Sanity check in the printout: each day's `leakage_dropped`
must now include `attempted_category`.

In [7]:
processed_stats = []
for day in DAY_ORDER:
    src = INTERIM / f'{day}.parquet'
    out = PROCESSED / f'{day}.parquet'
    if out.exists():
        print(f'[skip] {out.name} exists'); continue
    if not src.exists():
        print(f'!! missing interim {day}'); continue

    df = pd.read_parquet(src)
    n0 = len(df)

    label_col = 'label' if 'label' in df.columns else \
                [cn for cn in df.columns if 'label' in cn][0]
    category, attempted, label = parse_label(df[label_col])

    leak = [cn for cn in df.columns if cn in LEAKAGE_COLUMN_PATTERNS]
    assert 'attempted_category' in leak or 'attempted_category' not in df.columns, \
        'attempted_category present but not being dropped — Section 2 not re-run?'
    feat_cols = [cn for cn in df.columns
                 if cn not in leak + [label_col, 'source_file']]

    X = df[feat_cols].apply(pd.to_numeric, errors='coerce').astype('float32')
    keep = ~X.isna().any(axis=1)
    n_nan = int((~keep).sum())

    proc = X[keep].copy()
    proc['label'] = label[keep].values
    proc['attack_category'] = pd.Categorical(category[keep].values)
    proc['attempted'] = attempted[keep].values
    proc['source_file'] = pd.Categorical(df.loc[keep, 'source_file'].values)

    if len(proc) > MAX_ROWS_PER_DAY:
        frac = MAX_ROWS_PER_DAY / len(proc)
        proc = (proc.groupby('attack_category', observed=True, group_keys=False)
                    .apply(lambda g: g.sample(max(1, int(round(len(g)*frac))),
                                              random_state=SEED)))

    proc.to_parquet(out, compression='snappy', index=False)
    stats = {'day': day, 'rows_in': n0, 'rows_dropped_nan': n_nan,
             'rows_out': len(proc), 'leakage_dropped': leak,
             'n_features': len(feat_cols),
             'attack_rate': float(proc['label'].mean()),
             'attempted_rate': float(proc['attempted'].mean()),
             'categories': proc['attack_category'].value_counts().to_dict()}
    REPORT['days'][day] = stats
    processed_stats.append(stats)
    print(f"{day}: {n0:,} -> {len(proc):,} | features {len(feat_cols)} "
          f"| attack {stats['attack_rate']:.3f} | attempted {stats['attempted_rate']:.3f} "
          f"| dropped: {leak}")
    del df, proc, X

/tmp/ipykernel_1489/1599827031.py:36: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(max(1, int(round(len(g)*frac))),


wed_14_02: 5,898,350 -> 1,500,000 | features 82 | attack 0.049 | attempted 0.033 | dropped: ['id', 'flow_id', 'src_ip', 'src_port', 'dst_ip', 'dst_port', 'timestamp', 'attempted_category']


/tmp/ipykernel_1489/1599827031.py:36: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(max(1, int(round(len(g)*frac))),


thu_15_02: 5,410,102 -> 1,499,999 | features 82 | attack 0.007 | attempted 0.001 | dropped: ['id', 'flow_id', 'src_ip', 'src_port', 'dst_ip', 'dst_port', 'timestamp', 'attempted_category']


/tmp/ipykernel_1489/1599827031.py:36: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(max(1, int(round(len(g)*frac))),


...

wed_28_02: 6,568,726 -> 1,500,000 | features 82 | attack 0.008 | attempted 0.000 | dropped: ['id', 'flow_id', 'src_ip', 'src_port', 'dst_ip', 'dst_port', 'timestamp', 'attempted_category']


/tmp/ipykernel_1489/1599827031.py:36: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(max(1, int(round(len(g)*frac))),


thu_01_03: 6,551,401 -> 1,500,001 | features 82 | attack 0.006 | attempted 0.000 | dropped: ['id', 'flow_id', 'src_ip', 'src_port', 'dst_ip', 'dst_port', 'timestamp', 'attempted_category']


/tmp/ipykernel_1489/1599827031.py:36: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(max(1, int(round(len(g)*frac))),


fri_02_03: 6,311,371 -> 1,500,000 | features 82 | attack 0.023 | attempted 0.000 | dropped: ['id', 'flow_id', 'src_ip', 'src_port', 'dst_ip', 'dst_port', 'timestamp', 'attempted_category']


## 6. Data card + conversion report (MD documentation)

Overwrites the v1 report; the A1.0 deviation is stated explicitly so the audit
trail shows the fix rather than hiding it.

In [8]:
rows = []
for day in DAY_ORDER:
    s = REPORT['days'].get(day)
    if not s: continue
    n_cat = len([k for k in s['categories'] if k.upper() != 'BENIGN'])
    rows.append({'day': day, 'flows': s['rows_out'],
                 'attack_rate': round(s['attack_rate'], 3),
                 'attack_categories': n_cat})
comp = pd.DataFrame(rows)
comp.to_csv(ROOT / 'reports' / 'tables' / '23_cse2018_composition.csv', index=False)
display(comp)

lines = ['# CSE-CIC-IDS2018 (improved) — Data Card & Conversion Report (v2)\n',
         f'Generated: {REPORT["generated"]}\n',
         '\n## Version note (A1.0)\n',
         '- v2 regeneration: `attempted_category` (improved-release label '
         'metadata) added to the leakage drop list after it was found in the '
         'v1 feature matrix. Feature count 83 -> 82. See PREREG Addendum A1.0.\n',
         '\n## Provenance\n',
         '- Source: improved/corrected CSE-CIC-IDS2018 release '
         '(Engelen-group format, matching our CICIDS2017_improved).\n',
         f'- Location: `{RAW_2018}`\n',
         '\n## Pre-registered decisions (PREREG_cse2018.md)\n',
         f'- Per-day cap: {MAX_ROWS_PER_DAY:,} rows, stratified by attack_category, seed {SEED}.\n',
         '- Binary label: BENIGN -> 0; all attacks incl. `*- Attempted` -> 1 '
         '(identical to notebook 03 rule for CICIDS2017).\n',
         '- Leakage columns dropped: ' + ', '.join(LEAKAGE_COLUMN_PATTERNS) + '.\n',
         '- inf -> NaN -> row dropped; features float32.\n',
         '\n## Schema check vs CICIDS2017\n',
         f"- shared features: {REPORT['schema_check']['shared']}\n",
         f"- 2018-only: {REPORT['schema_check']['only_2018']}\n",
         f"- 2017-only: {REPORT['schema_check']['only_2017']}\n",
         '\n## Per-day composition\n', comp.to_markdown(index=False), '\n']
for day, s in REPORT['days'].items():
    lines.append(f"\n### {day}\n- rows_in: {s['rows_in']:,} | nan-dropped: "
                 f"{s['rows_dropped_nan']:,} | rows_out: {s['rows_out']:,} "
                 f"| features: {s['n_features']}\n"
                 f"- categories: {s['categories']}\n")
DATA_CARD_PATH.write_text(''.join(lines))
REPORT_PATH.write_text(''.join(lines))
with open(REPORT_PATH.with_suffix('.json'), 'w') as f:
    json.dump(REPORT, f, indent=2, default=str)
print('Written:', DATA_CARD_PATH)

,day,flows,attack_rate,attack_categories
0,wed_14_02,1500000,0.049,2
1,thu_15_02,1499999,0.007,2
2,fri_16_02,1500000,0.258,2
3,tue_20_02,1500000,0.048,2
4,wed_21_02,1500000,0.156,2
5,thu_22_02,1500001,0.000,3
6,fri_23_02,1500000,0.000,3
7,wed_28_02,1500000,0.008,3
8,thu_01_03,1500001,0.006,3
9,fri_02_03,1500000,0.023,1


Written: /content/drive/MyDrive/phd_thesis/reports/advisor_notes/data_card_cse2018.md


## 7. Loader smoke test (with v2 leakage guard)

Same guard as notebook 23: forbidden-substring check + exact 82-feature count.

In [9]:
from importlib import reload
import src.data.loaders as L; reload(L)

X, y, meta = L.load_dataset('cse_cic_ids2018', day='wed_14_02')
assert set(meta.columns) == {'attack_category', 'attempted', 'source_file'}
assert y.dtype == 'int8' and X.isna().sum().sum() == 0

FORBIDDEN_SUBSTRINGS = ('label', 'attempted', 'attack', 'category')
leaked = [cn for cn in X.columns
          if any(s in cn.lower() for s in FORBIDDEN_SUBSTRINGS)]
assert not leaked, f'LABEL-ADJACENT COLUMNS IN FEATURES: {leaked}'
assert X.shape[1] == 82, f'expected 82 features, got {X.shape[1]}'
print('contract + leakage guard OK:', X.shape,
      '| attack rate', round(float(y.mean()), 3))

for day, Xd, yd, md_ in L.cse2018_day_stream():
    print(f'{day:10s} {len(Xd):>9,} rows  attack {yd.mean():.3f}')

contract + leakage guard OK: (1500000, 82) | attack rate 0.049
wed_14_02  1,500,000 rows  attack 0.049
thu_15_02  1,499,999 rows  attack 0.007
fri_16_02  1,500,000 rows  attack 0.258
tue_20_02  1,500,000 rows  attack 0.048
wed_21_02  1,500,000 rows  attack 0.156
thu_22_02  1,500,001 rows  attack 0.000
fri_23_02  1,500,000 rows  attack 0.000
wed_28_02  1,500,000 rows  attack 0.008
thu_01_03  1,500,001 rows  attack 0.006
fri_02_03  1,500,000 rows  attack 0.023


## 8. Log the finding

In [10]:
# from src.utils.documentation import log_finding
# log_finding(
#     title='CSE-CIC-IDS2018 v2 regeneration (A1.0: attempted_category dropped)',
#     context='00_setup/05_convert_cse2018 (v2)',
#     what_we_did='Added attempted_category to the leakage drop list, deleted '
#                 'v1 processed parquets, regenerated Stage 2 from interim.',
#     what_we_found='<per-day rows, 82 features everywhere, guard passing>',
#     why_it_matters='Feature matrix is now leakage-clean before any detector '
#                    'training on 2018; documented as PREREG deviation A1.0.',
# )